In [ ]:
# ถ้ายังไม่มี package ให้รันก่อน
# !pip install numpy pandas matplotlib scipy scikit-learn ipywidgets

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from sklearn.model_selection import train_test_split

import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

In [ ]:
DATA_ROOT = Path(r"D:\sup_ai\level2_hack3_dashboard\level2_hack3_model")

CLASSES = [
    "frame_hit",
    "off_sweet_spot",
    "sweet_spot",
]

records = []

for label in CLASSES:
    class_dir = DATA_ROOT / label
    wav_files = sorted(class_dir.glob("*.wav"))

    for wav_path in wav_files:
        records.append({
            "path": str(wav_path),
            "file": wav_path.name,
            "label": label,
        })

df = pd.DataFrame(records)

print("Total wav files:", len(df))
display(df["label"].value_counts())
display(df.head())

In [ ]:
train_parts = []
test_parts = []

for label, group in df.groupby("label"):
    train_g, test_g = train_test_split(
        group,
        test_size=0.2,
        random_state=42,
        shuffle=True,
    )

    train_parts.append(train_g)
    test_parts.append(test_g)

train_df = pd.concat(train_parts).sample(frac=1, random_state=42).reset_index(drop=True)
test_df = pd.concat(test_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print("Train:", len(train_df))
display(train_df["label"].value_counts())

print("Test:", len(test_df))
display(test_df["label"].value_counts())

train_df.to_csv(DATA_ROOT / "train_files.csv", index=False)
test_df.to_csv(DATA_ROOT / "test_files.csv", index=False)

print("Saved:")
print(DATA_ROOT / "train_files.csv")
print(DATA_ROOT / "test_files.csv")

In [ ]:
def load_wav_signal(path):
    sample_rate, signal = wavfile.read(path)

    # stereo -> mono
    if signal.ndim == 2:
        signal = signal.mean(axis=1)

    # normalize to float32 [-1, 1]
    signal = signal.astype(np.float32)
    max_abs = np.max(np.abs(signal)) if len(signal) else 1.0

    if max_abs > 0:
        signal = signal / max_abs

    duration_sec = len(signal) / sample_rate

    return sample_rate, signal, duration_sec


def plot_signal(path, title=None, max_seconds=None):
    sample_rate, signal, duration_sec = load_wav_signal(path)

    if max_seconds is not None:
        max_len = int(sample_rate * max_seconds)
        signal = signal[:max_len]

    time_axis = np.arange(len(signal)) / sample_rate

    plt.figure(figsize=(14, 4))
    plt.plot(time_axis, signal, linewidth=0.8)
    plt.axhline(0, color="black", linewidth=0.6, alpha=0.4)
    plt.title(title or Path(path).name)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.grid(alpha=0.25)
    plt.show()

    print("Sample rate:", sample_rate)
    print("Duration:", round(duration_sec, 3), "sec")
    print("Samples:", len(signal))
    print("Min:", round(float(signal.min()), 4))
    print("Max:", round(float(signal.max()), 4))

In [ ]:
dataset_dropdown = widgets.Dropdown(
    options=["all", "train", "test"],
    value="all",
    description="Dataset:",
)

label_dropdown = widgets.Dropdown(
    options=CLASSES,
    value=CLASSES[0],
    description="Class:",
)

file_dropdown = widgets.Dropdown(
    options=[],
    description="File:",
    layout=widgets.Layout(width="650px"),
)

max_seconds_slider = widgets.FloatSlider(
    value=2.0,
    min=0.2,
    max=10.0,
    step=0.1,
    description="Seconds:",
    continuous_update=False,
)

output = widgets.Output()


def get_active_df():
    if dataset_dropdown.value == "train":
        return train_df
    if dataset_dropdown.value == "test":
        return test_df
    return df


def refresh_file_options(*args):
    active_df = get_active_df()
    filtered = active_df[active_df["label"] == label_dropdown.value].copy()

    options = [
        (row["file"], row["path"])
        for _, row in filtered.iterrows()
    ]

    file_dropdown.options = options

    if options:
        file_dropdown.value = options[0][1]


def render_signal(*args):
    with output:
        clear_output(wait=True)

        if not file_dropdown.value:
            print("No file selected")
            return

        path = file_dropdown.value
        row = df[df["path"] == path].iloc[0]

        print("Label:", row["label"])
        print("File:", row["file"])
        print("Path:", path)

        plot_signal(
            path,
            title=f"{row['label']} / {row['file']}",
            max_seconds=max_seconds_slider.value,
        )

        display(Audio(filename=path))


dataset_dropdown.observe(refresh_file_options, names="value")
label_dropdown.observe(refresh_file_options, names="value")
file_dropdown.observe(render_signal, names="value")
max_seconds_slider.observe(render_signal, names="value")

refresh_file_options()

display(widgets.VBox([
    widgets.HBox([dataset_dropdown, label_dropdown]),
    file_dropdown,
    max_seconds_slider,
    output,
]))

render_signal()